# LeetCode 4070 — Minimum Rotations to Dial a Number I bằng Python

Link đề: https://leetcode.com/problems/minimum-rotations-to-dial-a-number-i/ · Độ khó: **Easy** · Chủ đề: String, Math, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minRotations(self, s: str) -> int:
        ...
```

Ghi chú: đề và các ví dụ lấy từ đề gốc (Weekly Contest 522, câu 1); mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng một lời giải độc lập tìm đường đi ngắn nhất (0-1 BFS) trên đồ thị trạng thái (vị trí chuỗi, vị trí con trỏ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có một **mặt quay số vòng tròn** gồm các chữ số `0..9` theo thứ tự, `0` kề `9`; con trỏ ban đầu ở `0`; mỗi lần quay con trỏ nhảy sang chữ số **kề** (theo một trong hai hướng); cần **quay số lần ít nhất** để quay (dial) lần lượt từng chữ số của `s`.
2. Biết **khoảng cách trên vòng tròn** giữa hai chữ số `a` và `b` là `min(|a − b|, 10 − |a − b|)`, vì có hai đường đi quanh vòng tròn có độ dài cộng lại bằng `10`.
3. Hiểu vì sao có thể **tính từng bước độc lập** rồi cộng lại (tham lam): sau khi quay số xong chữ số hiện tại, con trỏ **đứng đúng** ở chữ số đó, nên bước kế tiếp chỉ phụ thuộc cặp (chữ số hiện tại, chữ số kế).
4. Đi từ **mô phỏng từng lần quay** → **hai khoảng cách theo `%`** → **công thức `abs`** → **bảng khoảng cách 10 × 10 (BFS trên vòng tròn)**, và làm quen `int(ch)`, `map`, `abs`, `%`.
5. Tránh các lỗi kinh điển: quên đường vòng qua `9 → 0`, bắt đầu con trỏ ở `s[0]` thay vì `0`, quên cập nhật chữ số trước, dùng `9` thay vì `10` làm chu kỳ.

## 2. Đề bài & ví dụ

Cho chuỗi `s` độ dài `10` gồm các chữ số. Mặt quay số chứa các chữ số `0` đến `9` theo thứ tự và là **vòng tròn**, nên `0` và `9` kề nhau. Con trỏ ban đầu chỉ vào `0`.

Để quay số từng chữ số của `s` **theo thứ tự**, ta quay cho tới khi con trỏ chỉ vào chữ số đó. Mỗi lần quay chuyển con trỏ sang một chữ số **kề** (theo một trong hai hướng, tùy ý). Nếu con trỏ đã chỉ vào chữ số cần quay thì không cần quay.

Trả về **tổng số lần quay nhỏ nhất** để quay số mọi chữ số của `s`.

**Ví dụ 1**

- Input: `s = "0192837465"`
- Output: `25`
- Giải thích: các bước quay (từ → đến, số lần): `0→0` (0), `0→1` (1), `1→9` (2), `9→2` (3), `2→8` (4), `8→3` (5), `3→7` (4), `7→4` (3), `4→6` (2), `6→5` (1). Tổng `0+1+2+3+4+5+4+3+2+1 = 25`.

**Ví dụ 2**

- Input: `s = "1200210200"`
- Output: `12`
- Giải thích: các bước: `0→1` (1), `1→2` (1), `2→0` (2), `0→0` (0), `0→2` (2), `2→1` (1), `1→0` (1), `0→2` (2), `2→0` (2), `0→0` (0). Tổng `12`.

**Ràng buộc:** `s.length == 10`; `s` chỉ gồm các ký tự `'0'` đến `'9'`.

In [1]:
from typing import List
import random
from collections import deque
from itertools import pairwise

examples = [
    ("0192837465", 25),
    ("1200210200", 12),
]
for s, expected in examples:
    print(f"s={s!r}, kỳ vọng {expected}")

s='0192837465', kỳ vọng 25
s='1200210200', kỳ vọng 12


## 3. Kiến thức nền

### 3.1. Duyệt chuỗi và đổi ký tự số thành số nguyên

Chuỗi `s` gồm **ký tự**, ví dụ `'7'` (chuỗi) khác `7` (số). Phải đổi bằng `int('7') = 7` mới tính toán được. `map(int, s)` đổi từng ký tự của `s` và cho ra một dãy số (dùng với `for` hoặc `list`).

In [2]:
s = "0192837465"
print(s[1], type(s[1]))                 # 1 <class 'str'>
print(int(s[1]), type(int(s[1])))       # 1 <class 'int'>
print(list(map(int, s)))                # [0, 1, 9, 2, 8, 3, 7, 4, 6, 5]
try:
    print('1' - '0')                     # không trừ được hai chuỗi
except TypeError as e:
    print("TypeError:", e)

1 <class 'str'>
1 <class 'int'>
[0, 1, 9, 2, 8, 3, 7, 4, 6, 5]
TypeError: unsupported operand type(s) for -: 'str' and 'str'


### 3.2. Vòng tròn và hai hướng quay

Các chữ số `0..9` xếp quanh một vòng tròn. Từ `a` tới `b` có hai đường: đi theo chiều tăng (`a → a+1 → ... → b`, quay vòng sau `9` là `0`) hoặc theo chiều giảm. Tổng độ dài của hai đường là đúng `10` (một vòng đầy đủ). Vì vậy nếu một đường dài `d` thì đường kia dài `10 − d`.

In [3]:
ring = list(range(10))
a, b = 2, 8
up = [(a + k) % 10 for k in range(0, 5)]       # vài vị trí đầu theo chiều tăng
down = [(a - k) % 10 for k in range(0, 5)]     # vài vị trí đầu theo chiều giảm
print("chiều tăng từ 2:", up)       # [2, 3, 4, 5, 6]
print("chiều giảm từ 2:", down)     # [2, 1, 0, 9, 8]
print("từ 2 đến 8: tăng =", (8 - 2) % 10, "bước, giảm =", (2 - 8) % 10, "bước")   # 6 và 4

chiều tăng từ 2: [2, 3, 4, 5, 6]
chiều giảm từ 2: [2, 1, 0, 9, 8]
từ 2 đến 8: tăng = 6 bước, giảm = 4 bước


### 3.3. Toán tử `%` luôn cho kết quả không âm (với số chia dương)

`(b − a) % 10` là số lần quay **theo chiều tăng** để đi từ `a` tới `b`; `(a − b) % 10` là số lần quay **theo chiều giảm**. Python cho `%` với số chia dương luôn trả về `0..9`, kể cả khi `b − a` âm: `(-3) % 10 = 7`.

In [4]:
for a, b in [(0, 1), (1, 0), (0, 9), (9, 0), (3, 3), (2, 8)]:
    print(f"{a} -> {b}: tăng {(b - a) % 10:>2}, giảm {(a - b) % 10:>2}")

0 -> 1: tăng  1, giảm  9
1 -> 0: tăng  9, giảm  1
0 -> 9: tăng  9, giảm  1
9 -> 0: tăng  1, giảm  9
3 -> 3: tăng  0, giảm  0
2 -> 8: tăng  6, giảm  4


### 3.4. `abs` và hai đường đi: `d = |a − b|`

`abs(a − b)` là khoảng cách **không đi vòng** (đi qua các số giữa). Đường đi vòng còn lại là `10 − d`. Nên khoảng cách ngắn nhất là `min(d, 10 − d)`. Ví dụ `0` và `9`: `d = 9`, đường vòng `10 − 9 = 1` ngắn hơn.

In [5]:
for a, b in [(0, 9), (0, 5), (2, 8), (3, 3), (1, 6)]:
    d = abs(a - b)
    print(f"{a} <-> {b}: d = {d}, 10 - d = {10 - d}, ngắn nhất = {min(d, 10 - d)}")

0 <-> 9: d = 9, 10 - d = 1, ngắn nhất = 1
0 <-> 5: d = 5, 10 - d = 5, ngắn nhất = 5
2 <-> 8: d = 6, 10 - d = 4, ngắn nhất = 4
3 <-> 3: d = 0, 10 - d = 10, ngắn nhất = 0
1 <-> 6: d = 5, 10 - d = 5, ngắn nhất = 5


### 3.5. Ghép cặp liên tiếp: `zip` và `pairwise`

Ta cần các cặp `(chữ số trước, chữ số sau)` liên tiếp. Với danh sách `digits`, `zip(digits, digits[1:])` hoặc `itertools.pairwise(digits)` cho các cặp đó. Để bước đầu tiên là `(0, s[0])`, ta nối thêm `0` vào đầu: `[0] + digits`.

In [6]:
digits = [int(ch) for ch in "1200210200"]
chain = [0] + digits                      # con trỏ xuất phát ở 0
print(list(pairwise(chain))[:5])          # [(0, 1), (1, 2), (2, 0), (0, 0), (0, 2)]
print(list(zip(chain, chain[1:]))[:5])    # giống hệt

[(0, 1), (1, 2), (2, 0), (0, 0), (0, 2)]
[(0, 1), (1, 2), (2, 0), (0, 0), (0, 2)]


### 3.6. BFS trên đồ thị nhỏ để dựng bảng khoảng cách

Mặt quay số là **đồ thị vòng** có 10 đỉnh; mỗi đỉnh kề hai đỉnh. Khoảng cách ngắn nhất (số cạnh) từ mọi đỉnh tới mọi đỉnh tính được bằng BFS từ từng đỉnh. Ta lưu thành bảng `dist[a][b]` rồi mỗi lần chỉ tra bảng.

In [7]:
def build_table(m=10):
    table = []
    for src in range(m):
        d = [-1] * m
        d[src] = 0
        q = deque([src])
        while q:
            u = q.popleft()
            for v in ((u + 1) % m, (u - 1) % m):     # hai đỉnh kề trên vòng tròn
                if d[v] == -1:
                    d[v] = d[u] + 1
                    q.append(v)
        table.append(d)
    return table


table = build_table()
print(table[0])        # [0, 1, 2, 3, 4, 5, 4, 3, 2, 1]
print(table[2])        # [2, 1, 0, 1, 2, 3, 4, 5, 4, 3]

[0, 1, 2, 3, 4, 5, 4, 3, 2, 1]
[2, 1, 0, 1, 2, 3, 4, 5, 4, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm tay ví dụ 1: `s = "0192837465"`. Con trỏ xuất phát ở `0`. Với mỗi chữ số kế tiếp, chọn hướng quay ngắn hơn trên vòng tròn:

| bước | từ | đến | `|đến − từ|` | `10 − |đến − từ|` | quay ít nhất |
|---|---|---|---|---|---|
| 1 | 0 | 0 | 0 | 10 | 0 |
| 2 | 0 | 1 | 1 | 9 | 1 |
| 3 | 1 | 9 | 8 | 2 | 2 (đi vòng qua `0`) |
| 4 | 9 | 2 | 7 | 3 | 3 (đi `9→0→1→2`) |
| 5 | 2 | 8 | 6 | 4 | 4 |
| 6 | 8 | 3 | 5 | 5 | 5 |
| 7 | 3 | 7 | 4 | 6 | 4 |
| 8 | 7 | 4 | 3 | 7 | 3 |
| 9 | 4 | 6 | 2 | 8 | 2 |
| 10 | 6 | 5 | 1 | 9 | 1 |

Tổng `0 + 1 + 2 + 3 + 4 + 5 + 4 + 3 + 2 + 1 = 25`, đúng với đề. Chú ý ở bước 3 và 4, đường **vòng qua `9 ↔ 0`** ngắn hơn đường đi thẳng.

**Vì sao tính từng bước độc lập là đúng?** Sau khi quay số một chữ số, con trỏ **đứng đúng** ở chữ số đó. Vị trí này không phụ thuộc ta đã quay theo hướng nào. Nên bước tiếp theo chỉ phụ thuộc cặp `(chữ số vừa quay, chữ số kế)`, và việc tối thiểu từng bước cũng tối thiểu tổng.

In [8]:
s = "0192837465"
pre, total = 0, 0
for ch in s:
    cur = int(ch)
    d = abs(cur - pre)
    cost = min(d, 10 - d)
    print(f"{pre} -> {cur}: d={d}, 10-d={10 - d}, cost={cost}")
    total += cost
    pre = cur
print("tổng:", total)

0 -> 0: d=0, 10-d=10, cost=0
0 -> 1: d=1, 10-d=9, cost=1
1 -> 9: d=8, 10-d=2, cost=2
9 -> 2: d=7, 10-d=3, cost=3
2 -> 8: d=6, 10-d=4, cost=4
8 -> 3: d=5, 10-d=5, cost=5
3 -> 7: d=4, 10-d=6, cost=4
7 -> 4: d=3, 10-d=7, cost=3
4 -> 6: d=2, 10-d=8, cost=2
6 -> 5: d=1, 10-d=9, cost=1
tổng: 25


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng lần quay

**Ý tưởng.** Làm đúng như đề, nhưng **thử cả hai hướng**: với mỗi chữ số mục tiêu, cho một bản sao con trỏ quay theo chiều tăng từng bước (`(p + 1) % 10`) cho tới khi tới mục tiêu và đếm số bước; làm tương tự theo chiều giảm; lấy hướng ít bước hơn. Ta mô phỏng bằng vòng `while`, chưa dùng công thức nào.

In [9]:
def min_rotations_simulate(s: str) -> int:
    pointer = 0
    total = 0
    for ch in s:
        target = int(ch)
        # quay theo chiều tăng
        p, up_steps = pointer, 0
        while p != target:
            p = (p + 1) % 10
            up_steps += 1
        # quay theo chiều giảm
        p, down_steps = pointer, 0
        while p != target:
            p = (p - 1) % 10
            down_steps += 1
        total += min(up_steps, down_steps)
        pointer = target                      # con trỏ đứng đúng ở chữ số vừa quay
    return total


for s, expected in examples:
    print(f"s={s!r} -> {min_rotations_simulate(s)} (kỳ vọng {expected})")

s='0192837465' -> 25 (kỳ vọng 25)
s='1200210200' -> 12 (kỳ vọng 12)


**Độ phức tạp (Cách 1).** Mỗi chữ số cần tối đa `9` lần quay mỗi hướng; với `n = len(s)` chữ số: **Time `O(n · D)`** với `D = 10` là số chữ số trên mặt quay số, tức `O(n)` vì `D` hằng số. **Space `O(1)`**.

**Nút thắt.** Không có nút thắt đáng kể: `n = 10`. Nhưng ta lặp lại vòng `while` chỉ để **đếm** số bước mà công thức `%` cho ngay. Mục tiêu tiếp theo: bỏ vòng `while`, tính khoảng cách bằng phép toán.

## 6. Ý tưởng thuật toán / bước đột phá

**Khoảng cách trên vòng tròn có công thức đóng.** Từ `a` tới `b`:

- theo chiều tăng: `(b − a) % 10` bước;
- theo chiều giảm: `(a − b) % 10` bước;
- hai số này cộng lại đúng `10` (nếu `a ≠ b`) hoặc đều bằng `0` (nếu `a = b`).

Nên đáp án cho một bước là `min((b − a) % 10, (a − b) % 10)`, hay `min(d, 10 − d)` với `d = |a − b|`.

**Tổng:** cộng khoảng cách của các cặp liên tiếp, với con trỏ xuất phát ở `0`:

```
answer = Σ dist(prev, cur)       với prev = 0 ở bước đầu, rồi prev = chữ số trước
```

Vì quay xong con trỏ đứng đúng chữ số, đây chính là tối ưu toàn cục (xem mục "Vì sao đúng").

In [10]:
for a, b in [(0, 1), (1, 9), (9, 2), (6, 5), (3, 3)]:
    up, down = (b - a) % 10, (a - b) % 10
    d = abs(a - b)
    assert min(up, down) == min(d, 10 - d)
    print(f"{a} -> {b}: tăng {up}, giảm {down}, ngắn nhất {min(up, down)}")

0 -> 1: tăng 1, giảm 9, ngắn nhất 1
1 -> 9: tăng 8, giảm 2, ngắn nhất 2
9 -> 2: tăng 3, giảm 7, ngắn nhất 3
6 -> 5: tăng 9, giảm 1, ngắn nhất 1
3 -> 3: tăng 0, giảm 0, ngắn nhất 0


## 7. Các cách tiếp theo

### Cách 2 — Hai khoảng cách bằng `%`

**Giải thích.** Với mỗi cặp `(pre, cur)`, tính `up = (cur − pre) % 10` và `down = (pre − cur) % 10`, cộng `min(up, down)`; cập nhật `pre = cur`.

In [11]:
def min_rotations_mod(s: str) -> int:
    total = 0
    pre = 0
    for ch in s:
        cur = int(ch)
        up = (cur - pre) % 10            # số lần quay theo chiều tăng
        down = (pre - cur) % 10          # số lần quay theo chiều giảm
        total += min(up, down)
        pre = cur
    return total


for s, expected in examples:
    print(f"s={s!r} -> {min_rotations_mod(s)} (kỳ vọng {expected})")

s='0192837465' -> 25 (kỳ vọng 25)
s='1200210200' -> 12 (kỳ vọng 12)


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — Công thức `min(d, 10 − d)` với `d = |cur − pre|` (cách của lời giải tham khảo)

**Giải thích.** Cùng ý với Cách 2 nhưng dùng `abs`: `d = abs(cur − pre)` là đường không đi vòng; `10 − d` là đường đi vòng. Dùng `map(int, s)` để duyệt các chữ số.

In [12]:
def min_rotations_abs(s: str) -> int:
    ans = pre = 0
    for cur in map(int, s):
        diff = abs(cur - pre)
        ans += min(diff, 10 - diff)
        pre = cur
    return ans


for s, expected in examples:
    print(f"s={s!r} -> {min_rotations_abs(s)} (kỳ vọng {expected})")

s='0192837465' -> 25 (kỳ vọng 25)
s='1200210200' -> 12 (kỳ vọng 12)


In [13]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def minRotations(self, s: str) -> int:
        ans = pre = 0
        for cur in map(int, s):
            diff = abs(cur - pre)
            ans += min(diff, 10 - diff)
            pre = cur
        return ans

**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Bảng khoảng cách 10 × 10 (BFS trên vòng tròn) và `pairwise`

**Giải thích.** Dựng bảng `table[a][b]` bằng BFS (mục 3.6), rồi cộng `table[a][b]` cho mọi cặp liên tiếp của `[0] + digits`. Không cần công thức nào; chỉ cần biết "khoảng cách là số cạnh ngắn nhất trên đồ thị vòng". Dễ mở rộng cho mặt quay số bất kỳ (ví dụ đồ thị không còn là vòng tròn đều).

In [14]:
def min_rotations_table(s: str) -> int:
    table = build_table(10)
    digits = [0] + [int(ch) for ch in s]
    return sum(table[a][b] for a, b in pairwise(digits))


for s, expected in examples:
    print(f"s={s!r} -> {min_rotations_table(s)} (kỳ vọng {expected})")

s='0192837465' -> 25 (kỳ vọng 25)
s='1200210200' -> 12 (kỳ vọng 12)


**Độ phức tạp Cách 4:** dựng bảng `O(D²)` (10 lần BFS mỗi lần `O(D)`), cộng `n` lần tra bảng `O(1)`: **Time `O(D² + n)`**, **Space `O(D²)`** với `D = 10`.

> Cách 3 là cách nộp bài gọn nhất. Cách 1 và 4 nhấn mạnh "khoảng cách trên vòng tròn" theo định nghĩa. Cách 2 giúp hiểu vì sao `10 − d` xuất hiện.

## 8. TRACE từng bước

Ta theo dõi Cách 3 trên `s = "1200210200"` (ví dụ 2), in `pre`, `cur`, `diff`, hai đường, chi phí và tổng tích lũy. Rồi Cách 1 mô phỏng từng lần quay trên một bước cụ thể `9 → 2`.

In [15]:
def trace_abs(s):
    print(f"s={s!r}")
    ans = pre = 0
    for step, cur in enumerate(map(int, s), start=1):
        diff = abs(cur - pre)
        cost = min(diff, 10 - diff)
        ans += cost
        print(f"  bước {step:>2}: {pre} -> {cur} | diff={diff}, vòng={10 - diff}, chi phí={cost}, tổng={ans}")
        pre = cur
    print("  => đáp án:", ans)
    return ans


assert trace_abs("1200210200") == 12

s='1200210200'
  bước  1: 0 -> 1 | diff=1, vòng=9, chi phí=1, tổng=1
  bước  2: 1 -> 2 | diff=1, vòng=9, chi phí=1, tổng=2
  bước  3: 2 -> 0 | diff=2, vòng=8, chi phí=2, tổng=4
  bước  4: 0 -> 0 | diff=0, vòng=10, chi phí=0, tổng=4
  bước  5: 0 -> 2 | diff=2, vòng=8, chi phí=2, tổng=6
  bước  6: 2 -> 1 | diff=1, vòng=9, chi phí=1, tổng=7
  bước  7: 1 -> 0 | diff=1, vòng=9, chi phí=1, tổng=8
  bước  8: 0 -> 2 | diff=2, vòng=8, chi phí=2, tổng=10
  bước  9: 2 -> 0 | diff=2, vòng=8, chi phí=2, tổng=12
  bước 10: 0 -> 0 | diff=0, vòng=10, chi phí=0, tổng=12
  => đáp án: 12


In [16]:
def trace_one_move(pointer, target):
    print(f"quay từ {pointer} đến {target}:")
    p, up = pointer, 0
    path_up = [p]
    while p != target:
        p = (p + 1) % 10
        up += 1
        path_up.append(p)
    p, down = pointer, 0
    path_down = [p]
    while p != target:
        p = (p - 1) % 10
        down += 1
        path_down.append(p)
    print(f"  chiều tăng : {path_up} -> {up} lần")
    print(f"  chiều giảm : {path_down} -> {down} lần")
    print(f"  chọn {min(up, down)}")
    return min(up, down)


assert trace_one_move(9, 2) == 3
assert trace_one_move(1, 9) == 2

quay từ 9 đến 2:
  chiều tăng : [9, 0, 1, 2] -> 3 lần
  chiều giảm : [9, 8, 7, 6, 5, 4, 3, 2] -> 7 lần
  chọn 3
quay từ 1 đến 9:
  chiều tăng : [1, 2, 3, 4, 5, 6, 7, 8, 9] -> 8 lần
  chiều giảm : [1, 0, 9] -> 2 lần
  chọn 2


Bảng tóm tắt (Cách 3, ví dụ 2: `s = "1200210200"`):

| bước | `pre → cur` | `diff` | `10 − diff` | chi phí | tổng |
|---|---|---|---|---|---|
| 1 | `0 → 1` | 1 | 9 | 1 | 1 |
| 2 | `1 → 2` | 1 | 9 | 1 | 2 |
| 3 | `2 → 0` | 2 | 8 | 2 | 4 |
| 4 | `0 → 0` | 0 | 10 | 0 | 4 |
| 5 | `0 → 2` | 2 | 8 | 2 | 6 |
| 6 | `2 → 1` | 1 | 9 | 1 | 7 |
| 7 | `1 → 0` | 1 | 9 | 1 | 8 |
| 8 | `0 → 2` | 2 | 8 | 2 | 10 |
| 9 | `2 → 0` | 2 | 8 | 2 | 12 |
| 10 | `0 → 0` | 0 | 10 | 0 | 12 |

Khi `diff = 0` thì `10 − diff = 10` nhưng `min(0, 10) = 0` nên chi phí đúng là `0` (không cần quay).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3. Đúng class `Solution`, đúng method `minRotations(self, s: str) -> int`.

In [17]:
class Solution:
    def minRotations(self, s: str) -> int:
        ans = 0
        pre = 0                                   # con trỏ ban đầu ở chữ số 0
        for cur in map(int, s):                   # từng chữ số cần quay, theo thứ tự
            diff = abs(cur - pre)                 # đường không đi vòng
            ans += min(diff, 10 - diff)           # chọn đường ngắn hơn trên vòng tròn
            pre = cur                             # quay xong, con trỏ đứng đúng ở cur
        return ans


for s, expected in examples:
    assert Solution().minRotations(s) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Khoảng cách trên vòng tròn:** mọi đường đi từ `a` tới `b` là dãy các bước ±1 (mod 10). Số bước ròng theo chiều tăng là `(b − a)` cộng một bội của `10`; ngắn nhất là một trong hai giá trị `(b − a) mod 10` hoặc `10 − ((b − a) mod 10)`; đó chính là `min(d, 10 − d)`. Đi vòng nhiều hơn một vòng chỉ làm tốn thêm.
- **Từng bước độc lập:** quay số chữ số thứ `k` yêu cầu con trỏ **đứng đúng** ở chữ số `s[k]` vào thời điểm đó. Vị trí con trỏ sau bước `k` được xác định hoàn toàn (`s[k]`), nên chi phí bước `k + 1` chỉ phụ thuộc `(s[k], s[k+1])`. Chi phí tổng là tổng các chi phí nhỏ nhất, và tối thiểu từng phần tối thiểu tổng (không có ràng buộc nào nối các bước ngoài vị trí đó).
- **Con trỏ xuất phát ở `0`:** bước đầu tiên là `(0, s[0])`, không phải `(s[0], s[0])`.
- **`diff = 0`:** hai chữ số bằng nhau cho chi phí `0` (`min(0, 10) = 0`).
- **Kiểm chứng độc lập:** ở mục test, ta dựng đồ thị trạng thái `(vị trí trong chuỗi, vị trí con trỏ)` và chạy 0-1 BFS (quay là chi phí 1, "đã ở đúng chữ số thì chuyển sang chữ số kế" chi phí 0) để xác nhận.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (đề cho `n = 10`) và `D = 10` là số chữ số trên mặt quay số.

- **Cách 1 (mô phỏng từng lần quay):** Time `O(n · D)`, Space `O(1)`.
- **Cách 2 (hai khoảng cách `%`):** Time `O(n)`, Space `O(1)`.
- **Cách 3 (`abs` và `min`):** Time `O(n)`, Space `O(1)`.
- **Cách 4 (bảng BFS):** Time `O(D² + n)`, Space `O(D²)`.
- Mọi cách đều không phụ thuộc cấu trúc băm hay ngẫu nhiên: trường hợp trung bình và xấu nhất như nhau. Không thể nhanh hơn `O(n)` vì phải đọc hết `s`.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` và viết một **lời giải độc lập** `oracle`: **0-1 BFS** trên đồ thị trạng thái `(k, p)` (`k` là số chữ số đã quay xong, `p` là vị trí con trỏ). Quay ±1 tốn `1`; nếu `p == s[k]` thì có cạnh chi phí `0` sang `(k + 1, p)`.

In [18]:
sol = Solution()
impls = {
    "simulate": min_rotations_simulate,
    "mod": min_rotations_mod,
    "abs": min_rotations_abs,
    "table": min_rotations_table,
    "submit": sol.minRotations,
}


def oracle(s):
    """0-1 BFS: trạng thái (k, p); quay +-1 chi phí 1; quay số khi p == s[k] chi phí 0."""
    n = len(s)
    INF = float("inf")
    dist = {(0, 0): 0}
    dq = deque([(0, 0)])
    best = INF
    while dq:
        k, p = dq.popleft()
        d = dist[(k, p)]
        if k == n:
            best = min(best, d)
            continue
        if p == int(s[k]):                                    # quay số: chi phí 0, đẩy lên đầu hàng đợi
            if dist.get((k + 1, p), INF) > d:
                dist[(k + 1, p)] = d
                dq.appendleft((k + 1, p))
        for q in ((p + 1) % 10, (p - 1) % 10):                # quay 1 lần: chi phí 1, đẩy xuống cuối
            if dist.get((k, q), INF) > d + 1:
                dist[(k, q)] = d + 1
                dq.append((k, q))
    return best


for s, expected in examples:
    assert oracle(s) == expected, (s, oracle(s), expected)
    for name, fn in impls.items():
        assert fn(s) == expected, (name, s, fn(s), expected)
print("PASS ví dụ của đề (mọi cách và oracle 0-1 BFS độc lập)")

PASS ví dụ của đề (mọi cách và oracle 0-1 BFS độc lập)


### 12.1. Các trường hợp biên

In [19]:
def check_all(s, expected, label):
    assert oracle(s) == expected, (label, "oracle", oracle(s), expected)
    for name, fn in impls.items():
        got = fn(s)
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_all("0000000000", 0, "toàn 0: con trỏ không phải quay")
check_all("5555555555", 5, "chỉ quay 5 lần ở bước đầu rồi đứng yên")
check_all("9999999999", 1, "0 -> 9 chỉ cần 1 lần (đi vòng), sau đó đứng yên")
check_all("1111111111", 1, "0 -> 1 một lần")
check_all("0123456789", 9, "tăng dần: 9 lần quay")
check_all("9876543210", 1 + 1 * 9, "giảm dần: 0 -> 9 (1 lần), rồi mỗi bước giảm 1")
check_all("5050505050", 50, "luân phiên 5 và 0: mỗi bước 5 lần, tối đa 10 x 5 = 50")
check_all("0505050505", 45, "bắt đầu bằng 0 (0 lần), rồi 9 bước, mỗi bước 5 lần")
check_all("1200210200", 12, "ví dụ 2 của đề")
check_all("6666666666", 4, "0 -> 6: đi vòng 4 lần ngắn hơn 6 lần")

PASS toàn 0: con trỏ không phải quay -> 0
PASS chỉ quay 5 lần ở bước đầu rồi đứng yên -> 5
PASS 0 -> 9 chỉ cần 1 lần (đi vòng), sau đó đứng yên -> 1
PASS 0 -> 1 một lần -> 1
PASS tăng dần: 9 lần quay -> 9
PASS giảm dần: 0 -> 9 (1 lần), rồi mỗi bước giảm 1 -> 10
PASS luân phiên 5 và 0: mỗi bước 5 lần, tối đa 10 x 5 = 50 -> 50
PASS bắt đầu bằng 0 (0 lần), rồi 9 bước, mỗi bước 5 lần -> 45
PASS ví dụ 2 của đề -> 12
PASS 0 -> 6: đi vòng 4 lần ngắn hơn 6 lần -> 4


### 12.2. Tính chất và phản ví dụ

In [20]:
random.seed(4070)


def random_s():
    return "".join(random.choice("0123456789") for _ in range(10))


# Tính chất 1: 0 <= đáp án <= 50 (mỗi bước tối đa 5); đáp án = 0 khi và chỉ khi s toàn '0'
for _ in range(500):
    s = random_s()
    a = sol.minRotations(s)
    assert 0 <= a <= 50
    assert (a == 0) == (s == "0" * 10)
print("PASS tính chất: 0 <= đáp án <= 50; bằng 0 chỉ khi s toàn '0'")

# Tính chất 2: phản xạ d -> (10 - d) % 10 không đổi đáp án (con trỏ cũng ở 0)
for _ in range(500):
    s = random_s()
    mirrored = "".join(str((10 - int(ch)) % 10) for ch in s)
    assert sol.minRotations(s) == sol.minRotations(mirrored)
print("PASS tính chất: phản xạ gương không đổi đáp án")

# Tính chất 3: bất đẳng thức tam giác từng bước: dist(a, c) <= dist(a, b) + dist(b, c)
def dist(a, b):
    d = abs(a - b)
    return min(d, 10 - d)

for a in range(10):
    for b in range(10):
        for c in range(10):
            assert dist(a, c) <= dist(a, b) + dist(b, c)
print("PASS tính chất: bất đẳng thức tam giác của khoảng cách vòng tròn")

# Phản ví dụ: bỏ qua đường vòng 9 <-> 0 (chỉ dùng abs)
def sai_chi_abs(s):
    total = pre = 0
    for ch in s:
        cur = int(ch)
        total += abs(cur - pre)
        pre = cur
    return total

print("0192837465 -> sai:", sai_chi_abs("0192837465"), "| đúng:", sol.minRotations("0192837465"))
assert sai_chi_abs("0192837465") != 25

PASS tính chất: 0 <= đáp án <= 50; bằng 0 chỉ khi s toàn '0'
PASS tính chất: phản xạ gương không đổi đáp án
PASS tính chất: bất đẳng thức tam giác của khoảng cách vòng tròn
0192837465 -> sai: 37 | đúng: 25


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle 0-1 BFS độc lập

In [21]:
random.seed(2026)
for trial in range(2500):
    s = random_s()
    expected = oracle(s)
    for name, fn in impls.items():
        assert fn(s) == expected, (name, s, fn(s), expected)

# thêm: các chuỗi chỉ gồm vài chữ số (nhiều cặp bằng nhau / đối xứng)
for trial in range(500):
    alphabet = random.sample("0123456789", random.randint(1, 3))
    s = "".join(random.choice(alphabet) for _ in range(10))
    expected = oracle(s)
    for name, fn in impls.items():
        assert fn(s) == expected, (name, s)
print("PASS 3000 input ngẫu nhiên: mọi cách khớp oracle 0-1 BFS độc lập")

PASS 3000 input ngẫu nhiên: mọi cách khớp oracle 0-1 BFS độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên đường đi vòng `9 ↔ 0`

In [22]:
def sai_quen_vong(s):
    total = pre = 0
    for ch in s:
        cur = int(ch)
        total += abs(cur - pre)            # chỉ tính đường không vòng
        pre = cur
    return total


print("sai:", sai_quen_vong("0192837465"), "| đúng:", Solution().minRotations("0192837465"))
print("sai:", sai_quen_vong("9999999999"), "| đúng:", Solution().minRotations("9999999999"))

sai: 37 | đúng: 25
sai: 9 | đúng: 1


**Vì sao sai:** mặt quay số là **vòng tròn**: `0` kề `9`. Đi từ `0` tới `9` chỉ cần `1` lần (qua cạnh `0-9`) chứ không phải `9`. Phải lấy `min(d, 10 − d)`.

### Lỗi 2: Bắt đầu con trỏ ở `s[0]` thay vì `0`

In [23]:
def sai_bat_dau_s0(s):
    total = 0
    pre = int(s[0])                        # sai: đề nói con trỏ ban đầu ở 0
    for ch in s:
        cur = int(ch)
        d = abs(cur - pre)
        total += min(d, 10 - d)
        pre = cur
    return total


print("sai:", sai_bat_dau_s0("1200210200"), "| đúng:", Solution().minRotations("1200210200"))

sai: 11 | đúng: 12


**Vì sao sai:** con trỏ **luôn bắt đầu ở `0`**, nên chữ số đầu tiên của `s` cũng tốn quay (trừ khi nó là `0`). Ví dụ 2 bắt đầu bằng `1`: bước `0 → 1` tốn `1` lần mà code sai bỏ qua, cho `11` thay vì `12`.

### Lỗi 3: Quên cập nhật `pre`

In [24]:
def sai_quen_pre(s):
    total = 0
    pre = 0
    for ch in s:
        cur = int(ch)
        d = abs(cur - pre)
        total += min(d, 10 - d)
        # quên: pre = cur
    return total


print("sai:", sai_quen_pre("0192837465"), "| đúng:", Solution().minRotations("0192837465"))

sai: 25 | đúng: 25


**Vì sao sai:** nếu không cập nhật `pre = cur` thì mọi chữ số đều được so với `0`, tức là ta tính khoảng cách từ `0` tới mỗi chữ số thay vì giữa các chữ số liên tiếp. Con trỏ phải **ở lại** chữ số vừa quay.

### Lỗi 4: Dùng `9` làm chu kỳ thay vì `10`

In [25]:
def sai_chu_ky_9(s):
    total = pre = 0
    for ch in s:
        cur = int(ch)
        d = abs(cur - pre)
        total += min(d, 9 - d)             # sai: có 10 chữ số, chu kỳ là 10
        pre = cur
    return total


print("sai:", sai_chu_ky_9("0192837465"), "| đúng:", Solution().minRotations("0192837465"))

sai: 21 | đúng: 25


**Vì sao sai:** các chữ số `0..9` là **10** vị trí trên vòng tròn, nên một vòng đầy đủ là `10` lần quay, và `0` kề `9` nghĩa là `9 → 0` mất **1** lần. Dùng `9 − d` coi vòng tròn chỉ có 9 vị trí, cho kết quả sai (khi `d = 9` còn ra `0` thay vì `1`).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**vòng tròn / quay vòng / hai chiều**" của các giá trị `0..m−1`: khoảng cách ngắn nhất giữa `a` và `b` là **`min(|a − b|, m − |a − b|)`**.
- Bài chọn "đường ngắn nhất trong hai hướng" cho từng đoạn, mà trạng thái sau mỗi đoạn là **cố định** → cộng độc lập (tham lam đúng).
- Nhớ điểm bắt đầu của con trỏ: dùng `pre = 0`, không phải phần tử đầu tiên của đầu vào.
- `map(int, s)` là cách gọn để duyệt chữ số của một chuỗi số.
- Khi không chắc tham lam đúng, kiểm tra bằng **BFS/Dijkstra trên đồ thị trạng thái** nhỏ (ở đây `(vị trí chuỗi, vị trí con trỏ)`) như oracle trong test.
- Mặt quay số kiểu điện thoại cổ hoặc khóa số: cùng khung này với chu kỳ `m` bất kỳ.

## 15. Biến thể của bài

1. **Mặt quay số có `m` chữ số** (thay vì 10) và vị trí xuất phát tùy ý.
2. **Chỉ được quay theo một hướng** (chỉ tăng): chi phí mỗi bước là `(cur − pre) % 10`.
3. **Trả về từng chi phí** của mỗi bước (danh sách).
4. **Chuỗi dài** (không cố định 10 ký tự): cùng thuật toán, `O(n)`.
5. **Có những chữ số bị "kẹt"** (không đi qua được): cần BFS thật sự trên đồ thị còn lại.

In [26]:
def min_rotations_general(s: str, m: int = 10, start: int = 0) -> int:
    """Biến thể 1 + 4: mặt quay số m vị trí, xuất phát ở start, chuỗi độ dài bất kỳ."""
    total, pre = 0, start
    for ch in s:
        cur = int(ch) % m
        d = abs(cur - pre)
        total += min(d, m - d)
        pre = cur
    return total


def min_rotations_one_way(s: str) -> int:
    """Biến thể 2: chỉ quay theo chiều tăng."""
    total, pre = 0, 0
    for ch in s:
        cur = int(ch)
        total += (cur - pre) % 10
        pre = cur
    return total


def step_costs(s: str) -> list[int]:
    """Biến thể 3."""
    out, pre = [], 0
    for cur in map(int, s):
        d = abs(cur - pre)
        out.append(min(d, 10 - d))
        pre = cur
    return out


assert min_rotations_general("0192837465") == 25
assert min_rotations_general("0192837465", m=10, start=0) == Solution().minRotations("0192837465")
assert min_rotations_general("0123", m=4) == 1 + 1 + 1
assert min_rotations_one_way("0192837465") == 1 + 8 + 3 + 6 + 5 + 4 + 7 + 2 + 9
assert step_costs("1200210200") == [1, 1, 2, 0, 2, 1, 1, 2, 2, 0]
assert sum(step_costs("1200210200")) == 12
long_s = "0192837465" * 10
assert min_rotations_general(long_s) == sum(step_costs(long_s))
print("PASS biến thể")

PASS biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng lần quay | quay +1 và quay −1 bằng `while`, lấy ít hơn | `O(n·D)` | `O(1)` | hiểu đề; đối chiếu |
| 2. Hai khoảng cách `%` | `min((b−a)%10, (a−b)%10)` | `O(n)` | `O(1)` | thấy vì sao có `10 − d` |
| 3. `min(d, 10 − d)` | `d = abs(cur − pre)` | `O(n)` | `O(1)` | **cách nộp bài** |
| 4. Bảng BFS 10 × 10 | khoảng cách trên đồ thị vòng | `O(D² + n)` | `O(D²)` | mặt quay số không đều / có hạn chế |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `int("7") + 1`, `(2 - 9) % 10`, `min(abs(3 - 8), 10 - abs(3 - 8))`, `Solution().minRotations("0000000000")`, `Solution().minRotations("9999999999")`.

**Bài 2 (tính tay).** Với `s = "3070000000"`: liệt kê từng bước `pre → cur` và chi phí, rồi tổng. Làm lại với `s = "8888888888"`.

**Bài 3 (điền TODO).** Hoàn thành `dial_todo(s)` theo Cách 3.

In [27]:
def dial_todo(s):
    total = 0
    pre = 0
    for ch in s:
        # TODO 1: cur = int(ch)
        # TODO 2: diff = abs(cur - pre)
        # TODO 3: total += min(diff, 10 - diff)
        # TODO 4: cập nhật pre
        pass
    return total

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `dial_table2(s)` theo Cách 4: dựng bảng khoảng cách bằng BFS trên đồ thị vòng 10 đỉnh rồi cộng. Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `dial_one_way(s)`: chỉ được quay theo chiều **tăng**. (b) Viết `dial_general(s, m, start)` cho mặt quay số `m` vị trí và con trỏ xuất phát ở `start`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `int("7")` là số `7`; `%` luôn không âm với số chia dương; `0` kề `9` nên `0 -> 9` là 1 lần. Bài 2: nhớ con trỏ bắt đầu ở `0`.
- **Mức 2 (vừa):** Bài 1: `8`, `3`, `5`, `0`, `1`. Bài 2: `3070000000`: `0→3` (3), `3→0` (3), `0→7` (3), `7→0` (3), rồi sáu chữ số `0` đều `0`; tổng `12`. `8888888888`: `0→8` mất `2` (đi vòng `0→9→8`), sau đó `0`; tổng `2`. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: BFS từ từng đỉnh `src` với hai đỉnh kề `(u + 1) % m`, `(u - 1) % m`; `table[a][b]` rồi `sum(table[a][b] for a, b in pairwise([0] + digits))`. Bài 5(a): cộng `(cur - pre) % 10`. Bài 5(b): thay `10` bằng `m` và `pre = start`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert int("7") + 1 == 8
assert (2 - 9) % 10 == 3
assert min(abs(3 - 8), 10 - abs(3 - 8)) == 5
assert Solution().minRotations("0000000000") == 0
assert Solution().minRotations("9999999999") == 1

# Bài 2
assert step_costs("3070000000") == [3, 3, 3, 3, 0, 0, 0, 0, 0, 0]
assert Solution().minRotations("3070000000") == 12
assert Solution().minRotations("8888888888") == 2

# Bài 3
def dial_todo_ans(s):
    total = 0
    pre = 0
    for ch in s:
        cur = int(ch)
        diff = abs(cur - pre)
        total += min(diff, 10 - diff)
        pre = cur
    return total

for s, expected in examples:
    assert dial_todo_ans(s) == expected

# Bài 4
def dial_table2(s):
    m = 10
    table = []
    for src in range(m):
        d = [-1] * m
        d[src] = 0
        q = deque([src])
        while q:
            u = q.popleft()
            for v in ((u + 1) % m, (u - 1) % m):
                if d[v] == -1:
                    d[v] = d[u] + 1
                    q.append(v)
        table.append(d)
    digits = [0] + [int(ch) for ch in s]
    return sum(table[a][b] for a, b in pairwise(digits))

for s, expected in examples:
    assert dial_table2(s) == expected

# Bài 5(a)
def dial_one_way(s):
    total = pre = 0
    for ch in s:
        cur = int(ch)
        total += (cur - pre) % 10
        pre = cur
    return total

assert dial_one_way("0192837465") == min_rotations_one_way("0192837465")
assert dial_one_way("9999999999") == 9

# Bài 5(b)
def dial_general(s, m, start):
    total, pre = 0, start
    for ch in s:
        cur = int(ch) % m
        d = abs(cur - pre)
        total += min(d, m - d)
        pre = cur
    return total

assert dial_general("0192837465", 10, 0) == 25
assert dial_general("1212", 4, 0) == 1 + 1 + 1 + 1
assert dial_general("33", 4, 0) == 1
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (khoảng cách ngắn nhất trên vòng tròn, cộng các cặp liên tiếp)

```python
class Solution:
    def minRotations(self, s: str) -> int:
        ans = pre = 0
        for cur in map(int, s):
            diff = abs(cur - pre)
            ans += min(diff, 10 - diff)
            pre = cur
        return ans
```

**Checklist**

- [ ] Con trỏ **bắt đầu ở `0`**, không phải ở `s[0]`?
- [ ] Mỗi bước dùng `min(diff, 10 − diff)` (có đường **đi vòng**)?
- [ ] Chu kỳ là **`10`** (các chữ số `0..9`), không phải `9`?
- [ ] Cập nhật `pre = cur` sau mỗi bước?
- [ ] Đổi ký tự thành số bằng `int(...)` / `map(int, s)` trước khi trừ?
- [ ] Đã thử: toàn `0`, toàn `9`, tăng dần, giảm dần, luân phiên `5` và `0`?
- [ ] Đã đối chiếu với mô phỏng hoặc BFS trên input ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def minRotations(self, s: str) -> int`?